# 별칭 2건 적용·삿포로 실제 발행 검증

읽기 전용 감사와 승인 파일의 지문을 확인한다. 수집·승인·발행·삭제는 실행하지 않는다. 이전 검토 23건과 삿포로 신규 103행은 다른 모집단이다.


In [1]:
import json,subprocess,tempfile,datetime
from pathlib import Path
repo=Path.cwd().resolve()
while not (repo/'scripts/tourism/audit.ts').exists():
    if repo==repo.parent:raise RuntimeError('Book해도 저장소 내부에서 실행하세요')
    repo=repo.parent
root=repo/'docs/research'
a=json.loads((root/'tourism-after-alias-sapporo-audit-20261002.json').read_text())
b=json.loads((root/'tourism-review-baseline-20261002.json').read_text())
action=json.loads((root/'tourism-alias-sapporo-20261002.json').read_text())
p={r['sourceId']:r for r in a['profile']};old={r['sourceId']:r for r in b['profile']}
assert a['counts']['catalogPlaces']==20810
assert (p['hokuto-places']['linkedFacilities'],p['sapporo-places']['facilities'],p['sapporo-places']['linkedFacilities'])==(4,103,15)
assert p['hokuto-places']['fetchedAt']==old['hokuto-places']['fetchedAt']
assert p['hokuto-places']['contentSha256']==old['hokuto-places']['contentSha256']
assert p['hokuto-places']['snapshotId']!=old['hokuto-places']['snapshotId']
for sid in ['furano-places','furano-events','eniwa-events']:assert p[sid]['snapshotId']==old[sid]['snapshotId']
print(json.dumps({'catalogPlaces':20810,'hokutoLinkedBefore':2,'hokutoLinkedAfter':4,'sapporoRecords':103,'sapporoLinked':15,'originalFetchTimePreserved':True},ensure_ascii=False,indent=2))


{
  "catalogPlaces": 20810,
  "hokutoLinkedBefore": 2,
  "hokutoLinkedAfter": 4,
  "sapporoRecords": 103,
  "sapporoLinked": 15,
  "originalFetchTimePreserved": true
}


In [2]:
from collections import Counter
key=lambda r:(r['sourceId'],r['externalId'])
original={key(r) for r in b['diagnostics'] if r['placeId'] is None}
remaining=[r for r in a['diagnostics'] if key(r) in original and r['placeId'] is None]
assert len(original)==23 and len(remaining)==21
srows=[r for r in a['diagnostics'] if r['sourceId']=='sapporo-places']
assert len(srows)==len({r['externalId'] for r in srows})==103
assert all(isinstance(r['externalId'],str) and r['externalId'].startswith('0') for r in srows)
assert Counter(r['reason'] for r in srows if r['placeId'] is None)=={'NAME_MISMATCH':87,'REGION_MISMATCH':1}
aliases=json.loads((repo/'ops/tourism/approved-aliases.json').read_text())
for approved in [r for r in aliases if r['sourceId']=='hokuto-places']:
    match=next(r for r in a['diagnostics'] if key(r)==(approved['sourceId'],approved['externalId']))
    assert match['placeId']==approved['placeId'] and approved['catalogWebsite'] is None
assert action['sapporo']['collectorReport']['resources'][0]['rows']==103
assert action['aliasApplication']['freshCollectionSucceeded'] is False
print(json.dumps({'previousUnlinkedRemaining':21,'sapporoUnlinked':88,'sapporoUnlinkedReasons':action['sapporo']['unlinkedReasons'],'newHokutoCollectionSucceeded':False},ensure_ascii=False,indent=2))


{
  "previousUnlinkedRemaining": 21,
  "sapporoUnlinked": 88,
  "sapporoUnlinkedReasons": {
    "NAME_MISMATCH": 87,
    "REGION_MISMATCH": 1
  },
  "newHokutoCollectionSucceeded": false
}


## 現在 DB 再감사

Node 의존성과 로컬 PostgreSQL 설정이 필요하다. `.env` 비밀값은 출력하지 않는다. 다른 작업이 snapshot을 바꿨다면 비교는 실패할 수 있다.


In [3]:
RUN_LIVE_AUDIT=True
if RUN_LIVE_AUDIT:
    with tempfile.TemporaryDirectory() as tmp:
        output=Path(tmp)/'audit.json'
        subprocess.run(['npm','run','tourism:audit','--','--date','2026-10-02','--output',str(output)],cwd=repo,capture_output=True,text=True,check=True)
        live=json.loads(output.read_text())
    assert live['counts']==a['counts']
    assert {(r['sourceId'],r['snapshotId'],r['linkedFacilities']) for r in live['profile']}=={(r['sourceId'],r['snapshotId'],r['linkedFacilities']) for r in a['profile']}
    print(json.dumps({'checkedAt':live['checkedAt'],'databaseMatchesPublishedAudit':True,'catalogPlaces':live['counts']['catalogPlaces']},ensure_ascii=False,indent=2))
    print(a['sql'])
else:print('Live audit skipped.')


{
  "checkedAt": "2026-10-02T00:46:46.210Z",
  "databaseMatchesPublishedAudit": true,
  "catalogPlaces": 20810
}
SELECT s.id AS "sourceId",v.id AS "snapshotId",
v.content_sha256 AS "contentSha256",v.fetched_at AS "fetchedAt",v.published_at AS "publishedAt",
s.enabled,s.rights_status AS "rightsStatus",v.record_count AS "declaredRecords",
count(r.external_id)::int AS records,
count(*) FILTER (WHERE r.kind='place')::int AS facilities,
count(*) FILTER (WHERE r.kind='event')::int AS events,
count(*) FILTER (WHERE r.kind='place' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "facilitiesWithCoordinates",
count(*) FILTER (WHERE r.kind='place' AND r.canonical_place_id IS NOT NULL)::int AS "linkedFacilities",
count(*) FILTER (WHERE r.kind='event' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "eventsWithCoordinates",
count(*) FILTER (WHERE r.kind='event' AND r.date_status IN ('confirmed','tentative') AND r.start_date<=$2::date AND r.end_date>=$2::date AND r.with

## 해석 범위

삿포로 15/103은 보수적인 catalog 연결률이며 시설의 존재 비율이나 추천 정확도가 아니다. 북토는 최신 수집 실패 후 원래 수집 시각을 유지한 재처리다. 실제 새 수집·운영 배포와 구분한다.
